In [ ]:
import time
import struct
from database import Database
import sys
import random

import chipwhisperer as cw

In [ ]:
def as_hex(s):
    return "".join([f"\\x{ord(c):02x}" for c in s])

def make_pkt(data):
    pkt = b"\x01"
    pkt += struct.pack('!H', len(data))
    pkt += data

    # Checksum
    chk = (256 - sum(pkt[1:])) & 0xff
    pkt += bytes([chk, 0x03])

    return pkt

In [ ]:
scope = cw.scope()

In [ ]:
scope.default_setup()
scope.io.tio1 = 'serial_tx'
scope.io.tio2 = 'serial_rx'
scope.io.tio3 = 'gpio_low'
scope.io.glitch_trig_mcx = 'glitch'

In [ ]:
# scope.clock.adc_mul = 0
# scope.clock.clkgen_src = 'system'
# scope.clock.clkgen_freq = 200e6


scope.clock.clkgen_src = 'system'
scope.clock.clkgen_freq = 100e6
scope.clock.adc_mul = 1

scope.gain.db = 40


target = cw.target(scope)
target.baud = 9600
target.flush()

scope.trigger.triggers = 'tio1'
scope.trigger.module = 'UART'
scope.gain.db = 12

scope.UARTTrigger.enabled = True
scope.UARTTrigger.baud = 9600
scope.UARTTrigger.set_pattern_match(0, '\x22\x03\x01\x00\x01\x00\xff\x03') # Synchronize
scope.UARTTrigger.trigger_source = 0

scope.glitch.enabled = True
scope.glitch.clk_src = "pll"

# scope.io.glitch_hp = False
# scope.io.glitch_lp = True

scope.io.glitch_hp = True
scope.io.glitch_lp = False
scope.io.glitch_trig_mcx = 'glitch'

scope.glitch.output = "enable_only"
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

In [ ]:
def read_block(target, block):
    start, end = block

    print("reading flash", hex(start), hex(end))
    pkt = make_pkt(b'\x15' + struct.pack('>II', start, end))

    target.write(pkt)   
    resp = target.read(6)
    if resp != '\x81\x00\x01\x15\xea\x03':
        target.read(1)
        print("wrong ack", as_hex(resp))
        return None

    rem = end - start
    dat = b""
    
    while rem > 0:
        target.write(b'\x81\x00\x01\x15\xea\x03')
        length = target.read(3)
        
        if len(length) != 3:
            print("wrong length received", as_hex(length))
            return None
        
        length = bytes([ord(c) for c in length])
        length = struct.unpack('!H', length[1:])[0] - 1 # Data starts with 0x15

        resp = b""
        while len(resp) < length + 3: 
            resp += bytes([ord(c) for c in target.read(target.in_waiting())])
            time.sleep(0.05)

        dat += resp[1:-2]
        rem -= length
    return dat

In [ ]:
gc = cw.GlitchController(groups=[], parameters=["ext_offset", "repeat"])
# min_delay = int(8.5e-6 * scope.clock.clkgen_freq)
# max_delay = int(9.2e-6 * scope.clock.clkgen_freq)

# min_delay = int(2.0e-6 * scope.clock.clkgen_freq)
# max_delay = int(10.0e-6 * scope.clock.clkgen_freq)

min_delay = int(5.0e-6 * scope.clock.clkgen_freq)
max_delay = int(6.5e-6 * scope.clock.clkgen_freq)

# min_delay = int(2.0e-6 * scope.clock.clkgen_freq)
# max_delay = int(8.0e-6 * scope.clock.clkgen_freq)

# min_delay = int(5.0e-6 * scope.clock.clkgen_freq)
# max_delay = int(5.8e-6 * scope.clock.clkgen_freq)

print(min_delay, max_delay)

gc.set_range("ext_offset", min_delay, max_delay)
gc.set_range("repeat", 50, 400) 
gc.set_global_step(10)
gc_id.set_step("repeat", 20)

gc.display_stats()
#gc.glitch_plot(plotdots={"success":"+r", "reset":"xy", "normal":'xg'})


In [ ]:
gc_id = cw.GlitchController(groups=[], parameters=["ext_offset", "repeat"])
# min_delay = int(8.5e-6 * scope.clock.clkgen_freq)
# max_delay = int(9.2e-6 * scope.clock.clkgen_freq)

# min_delay = int(1.5e-6 * scope.clock.clkgen_freq)
# max_delay = int(8.0e-6 * scope.clock.clkgen_freq)

# min_delay = int(6.1e-6 * scope.clock.clkgen_freq)
# max_delay = int(6.3e-6 * scope.clock.clkgen_freq)

# min_delay = int(5.5e-6 * scope.clock.clkgen_freq)
# max_delay = int(7.0e-6 * scope.clock.clkgen_freq)

# min_delay = int(875e-6 * scope.clock.clkgen_freq)
# max_delay = int(1040e-6 * scope.clock.clkgen_freq)

# min_delay = int(6.0e-6 * scope.clock.clkgen_freq)
# max_delay = int(6.5e-6 * scope.clock.clkgen_freq)

min_delay = int(5.0e-6 * scope.clock.clkgen_freq)
max_delay = int(7.0e-6 * scope.clock.clkgen_freq)

print(min_delay, max_delay)

gc_id.set_range("ext_offset", min_delay, max_delay)
gc_id.set_range("repeat", 50, 400) 

gc_id.set_global_step(10)
gc_id.set_step("ext_offset", 10)
gc_id.set_step("repeat", 20)

gc_id.display_stats()


In [ ]:
N_GLITCHES = 10

blocks = []

addr = 0x0
total_size = 0
for size in [0x2000] * 8 + [0x8000] * 46:
    blocks.append((addr, addr + size - 1))
    addr += size
    total_size += size

# addr = 0x80_0000
# for size in [0x8000] * 48:
#     blocks.append((addr, addr + size - 1))
#     addr += size
#     total_size += size

print(total_size / 1024 / 1024)
    
db = Database(sys.argv, extra_params=['delay', 'length'])

leave_reset = False

while True:
    for id_delay, id_width in gc_id.glitch_values(clear=False):
        # id_delay = id_delay[0]
        
        for delay, width in gc.glitch_values(clear=False):

            scope.glitch.ext_offset = delay
            scope.glitch.repeat = width
            scope.glitch.width = 0
            scope.glitch.offset = 0
    
            scope.UARTTrigger.set_pattern_match(0, '\x22\x03\x01\x00\x01\x00\xff\x03') # Synchronize
            scope.sc.arm(False)
            scope.arm()

            # target.write(make_pkt(b'\x30' + (b'\x00' * 16)))
        
            scope.io.tio3 = 'gpio_high' # Release Reset
            time.sleep(.2)
            target.read(target.in_waiting()) # Clean all bytes in serial input buffer
    
            try:
                # Sync
                target.write(b'\x00' * 10)
                assert target.read(1) == '\x00', "init 1 failed"
    
                # Connect
                target.write(b'\x55')
                assert target.read(1) == '\xc1', "init 2 failed"
    
                # Get device type
                target.write(b'\x01\x00\x01\x38\xc7\x03')
                resp = target.read(6)
                assert resp == '\x81\x00\x01\x38\xc7\x03', "Wrong Resp Device Type 1"
    
                target.write(b'\x81\x00\x01\x38\xc7\x03')
                resp = target.read(30)
                if resp != '\x81\x00\x19\x38\x10\x01\xff\x40\x00\x28\x02\x00\x01\x6e\x36\x00\x00\x7a\x12\x00\x07\x27\x0e\x00\x01\x7d\x78\x40\x92\x03':
                    print("unexpected resp device type 2", as_hex(resp))
    
                # Set up frequencies
                target.write(make_pkt(b'\x32\x00\x7a\x12\x00\x07\x27\x0e\x00'))
    
                resp = target.read(6)
                if resp != '\x81\x00\x01\x32\xcd\x03':
                    print("wrong resp freq 1", as_hex(resp))  
    
                target.write(b'\x81\x00\x01\x32\xcd\x03')
                resp = target.read(14)
                if resp != '\x81\x00\x09\x32\x07\x27\x0e\x00\x01\xc9\xc3\x80\x7c\x03':
                    print("unexpected resp freq 2", as_hex(resp))
    
                # Set bitrate (9600 baud)
                target.write(b'\x01\x00\x05\x34\x00\x00\x25\x80\x22\x03')
                resp = target.read(6)
                if resp != '\x81\x00\x01\x34\xcb\x03':
                    print("wrong resp bitrate 1", as_hex(resp))
    
                # Synchronize!
                target.write(b'\x01\x00\x01\x00\xff\x03')
                resp = target.read(7)
    
                if resp == '\x81\x00\x01\x00\xff\x03':
                    db.insert(0, color='O', response=b'Sync OK', delay=delay, length=width)

                    for _ in range(N_GLITCHES):
                        target.read(target.in_waiting()) # Clean all bytes in serial input buffer
                        
                        # Check auth mode
                        target.write(make_pkt(b'\x2c'))
                        resp = target.read(6)
                        target.write(b'\x81\x00\x01\x2c\xd3\x03')
                        resp = target.read(7)
                        print("Auth mode", as_hex(resp)) # \x81\x00\x02\x2c\x00\xd2\x03 = ID_AUTH_MODE

                        if len(resp) == 0:
                            break
                    
                        # Now we need to pass auth mode
                        scope.glitch.repeat = id_width
                        scope.glitch.ext_offset = id_delay
                        scope.UARTTrigger.set_pattern_match(0, '\x00\x00\x00\x00\x00\x00\xbf\x03')
                        # scope.UARTTrigger.set_pattern_match(0, '\x00\x00\x00\x00\x00\x00\x00\xbf') 
                        scope.sc.arm(False)
                        scope.arm()

                        # print("Sending ID")
                        target.write(make_pkt(b'\x30' + (b'\x00' * 16)))
                        resp = target.read(6)
        
                        # if resp == '\x81\x00\x02\x30\x00\xce\x03':
                        if resp == '\x81\x00\x01\x30\xcf\x03': # Manual doesn't match bootrom
                            print("ID bypass")
                            db.insert(0, color='R', response=b'ID OK', delay=id_delay, length=id_width)
                        elif resp == '\x81\x00\x02\xb0\xdb\x73': # Wrong ID
                            db.insert(0, color='G', response=b"ID: " + as_hex(resp).encode(), delay=id_delay, length=id_width)
                        elif resp == '':
                            db.insert(0, color='Y', response=b'ID', delay=id_delay, length=id_width)
                        else:
                            db.insert(0, color='Y', response=b"ID: " + as_hex(resp).encode(), delay=id_delay, length=id_width)

                        
    
                        id_ok = True
                        time.sleep(0.01)
                        extra = target.read(target.in_waiting()) # Clean all bytes in serial input buffer
                        print("Extra data", as_hex(extra))
                       
                        if id_ok:
                            for block in blocks:
                                resp = read_block(target, block)  
                                if resp is not None:
                                    leave_reset = True
                                    db.insert(0, color='R', response=b'Block OK', delay=id_delay, length=id_width)
                                    print("Got", block, len(resp))
                                    print(resp.hex())
                                    print(resp)
                                    with open(f'out_{block[0]:08x}.bin', 'wb') as f:
                                        f.write(resp)
                                else:
                                    break
                        
                        if leave_reset:        
                            raise RuntimeError("DONE!")

                        time.sleep(0.1)

                    break # Get new glitch parameters for 2nd glitch

                        
    
                elif resp == '\x81\x00\x02\x80\xdc\xa2\x03': # Programmer Disabled
                    db.insert(0, color='G', response=b"Sync: " + as_hex(resp).encode(), delay=delay, length=width)
                elif resp == '\x81\x00\x02\x80\xc1\xbd\x03': # Packet error
                    db.insert(0, color='Y', response=b"Sync: " + as_hex(resp).encode(), delay=delay, length=width)
                elif resp == '\x81\x00\x02\x80\xc3\xbb\x03': # Flow error
                    db.insert(0, color='Y', response=b"Sync: " + as_hex(resp).encode(), delay=delay, length=width) 
                elif resp == '':
                    db.insert(0, color='Y', response=b'Sync', delay=delay, length=width)
                else:
                    db.insert(0, color='Y', response=b"Sync: " + as_hex(resp).encode(), delay=delay, length=width)


            except AssertionError as e:
                print(delay, width, e)
                db.insert(0, color='Y', response=b'Assert ' + str(e).encode(), delay=delay, length=width)

            finally:
                if not leave_reset:
                    scope.io.tio3 = 'gpio_low' # Assert reset
                    time.sleep(.1)



In [ ]:
 for block in blocks:
    resp = read_block(target, block)  
    if resp is not None:
        print("Got", block, len(resp))
        print(resp.hex())
        print(resp)
        with open(f'out_{block[0]:08x}.bin', 'wb') as f:
            f.write(resp)

In [ ]:
# Check protection
target.read(target.in_waiting())
target.write(make_pkt(b'\x21'))
resp = target.read(7)

target.write('\x81\x00\x01\x21\xde\x03')

resp = target.read(10)
print(as_hex(resp))

In [ ]:
blocks = []

addr = 0x0
total_size = 0

for size in [0x2000] * 8 + [0x8000] * 46:
    blocks.append((addr, addr + size - 1))
    addr += size
    total_size += size

addr = 0x80_0000
for size in [0x8000] * 48:
    blocks.append((addr, addr + size - 1))
    addr += size
    total_size += size

In [ ]:
for a, s in blocks:
    print(hex(a))

In [ ]:
 for block in blocks:
    resp = read_block(target, block)  
    if resp is not None:
        print("Got", block, len(resp))
        print(resp.hex())
        print(resp)
        with open(f'out2/out_{block[0]:08x}.bin', 'wb') as f:
            f.write(resp)